# Notebook 04: App de Streamlit

**Entrada:** `models/model.pkl` (modelo entrenado en NB03)  
**Salida:** App web interactiva desplegada en Streamlit Community Cloud

Este notebook documenta la app y permite lanzarla localmente para pruebas.
El despliegue público para la presentación se hace desde Streamlit Community Cloud.

## Estructura de la app
- **Sidebar:** información del modelo (AUC-ROC, proteínas de entrenamiento, limitaciones)
- **Panel principal:** entrada de secuencias FASTA (texto o archivo)
- **Resultados:** gráfico de barras + tabla ordenada por score + descarga CSV

## Archivos del repositorio
```
antigen_predictor/
├── app/
│   └── app.py          ← código de la aplicación
├── models/
│   └── model.pkl       ← modelo entrenado en NB03
└── requirements.txt    ← dependencias para Streamlit Community Cloud
```

---
## 0. Verificación del entorno

In [1]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
MODEL_PATH   = PROJECT_ROOT / 'models' / 'model.pkl'
APP_PATH     = PROJECT_ROOT / 'app' / 'app.py'

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"model.pkl existe:  {MODEL_PATH.exists()} ({MODEL_PATH.stat().st_size / 1024:.1f} KB)" if MODEL_PATH.exists() else f"model.pkl existe:  False")
print(f"app.py existe:     {APP_PATH.exists()}")

Raíz del proyecto: /home/apa/Documentos/github/antigen_predictor
model.pkl existe:  True (5274.5 KB)
app.py existe:     False


---
## 1. Lanzar la app localmente

Desde el terminal, con el entorno virtual activado, ejecuta:

```bash
cd ~/Documentos/github/antigen_predictor/app
streamlit run app.py
```

La app se abre automáticamente en `http://localhost:8501`.  
Para detenerla pulsa `Ctrl+C` en el terminal.

También puedes lanzarla desde esta celda (se ejecuta en segundo plano):

In [3]:
import subprocess
import sys

app_dir = PROJECT_ROOT / 'app'

print("Lanzando Streamlit en http://localhost:8501 ...")
print("Para detenerlo: interrumpe el kernel o cierra el terminal.")
print()

# Lanza streamlit como proceso hijo
# Nota: esto bloquea la celda. Usa el terminal si prefieres tenerlo en background.
subprocess.run(
    [sys.executable, "-m", "streamlit", "run", "app.py"],
    cwd=str(app_dir)
)

Lanzando Streamlit en http://localhost:8501 ...
Para detenerlo: interrumpe el kernel o cierra el terminal.


      👋 Welcome to Streamlit!

      If you'd like to receive helpful onboarding emails, news, offers, promotions,
      and the occasional swag, please enter your email address below. Otherwise,
      leave this field blank.

      Email: 

2026-05-16 18:23:52.917 


CompletedProcess(args=['/home/apa/Documentos/github/antigen_predictor/.venv/bin/python3', '-m', 'streamlit', 'run', 'app.py'], returncode=255)

---
## 2. Verificación de la app

Una vez abierta la app en el navegador, prueba el flujo completo:

1. Haz clic en **"Cargar proteínas SARS-CoV-2"** para usar el ejemplo precargado
2. Haz clic en **"🔬 Predecir antigenicidad"**
3. Verifica que **Spike glycoprotein** aparece con el score más alto
4. Descarga el CSV de resultados
5. Prueba subir un archivo `.fasta` propio

Si Spike aparece arriba, el modelo captura algo biológicamente coherente
y la demo está lista para la presentación.

---
## 3. Despliegue en Streamlit Community Cloud

Streamlit Community Cloud permite desplegar la app desde GitHub de forma gratuita.
La app queda disponible en una URL pública permanente del tipo `tuapp.streamlit.app`,
accesible desde cualquier navegador sin necesidad de tener el ordenador encendido.

Esta es la opción que usaremos para la presentación en clase.

### 3.1 Preparar el repositorio

Antes de desplegar necesitamos un archivo `requirements.txt` en la **raíz del repositorio**
con las dependencias que Streamlit Community Cloud instalará.
Ejecuta la celda siguiente para generarlo automáticamente:

In [ ]:
requirements_content = """biopython
scikit-learn
joblib
matplotlib
pandas
numpy
streamlit
"""

req_path = PROJECT_ROOT / 'requirements.txt'

# Leer el contenido actual si existe
current = req_path.read_text() if req_path.exists() else ""

# Verificar si ya contiene las dependencias de la app
app_deps = ['biopython', 'scikit-learn', 'streamlit']
missing  = [d for d in app_deps if d not in current]

if not missing:
    print(f"requirements.txt ya contiene todas las dependencias necesarias.")
    print(f"Contenido actual:\n{current}")
else:
    req_path.write_text(requirements_content)
    print(f"requirements.txt actualizado en: {req_path}")
    print(f"Contenido:\n{requirements_content}")

### 3.2 Verificar que model.pkl está en el repositorio

Streamlit Community Cloud clona el repositorio de GitHub para desplegar la app.
El modelo debe estar incluido en el repositorio para que la app pueda cargarlo.

Comprueba que `model.pkl` **no está en `.gitignore`** y que está subido a GitHub:

In [ ]:
import subprocess

gitignore_path = PROJECT_ROOT / '.gitignore'
gitignore_content = gitignore_path.read_text() if gitignore_path.exists() else ""

if 'model.pkl' in gitignore_content:
    print("⚠️  model.pkl está en .gitignore — Streamlit Cloud no podrá descargarlo.")
    print("   Elimina 'model.pkl' del .gitignore y vuelve a hacer commit.")
else:
    print("✓ model.pkl no está en .gitignore")

# Comprobar si está trackeado por git
result = subprocess.run(
    ['git', 'ls-files', 'models/model.pkl'],
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True
)
if result.stdout.strip():
    print("✓ model.pkl está trackeado por git")
else:
    print("⚠️  model.pkl no está trackeado por git.")
    print("   Ejecuta en el terminal:")
    print("   git add models/model.pkl")
    print("   git commit -m 'Add trained model'")
    print("   git push")

### 3.3 Pasos para desplegar en Streamlit Community Cloud

Una vez que el repositorio está listo (model.pkl subido, requirements.txt actualizado):

1. Ve a [share.streamlit.io](https://share.streamlit.io) y crea una cuenta gratuita (puedes usar tu cuenta de GitHub)
2. Haz clic en **"New app"**
3. Selecciona tu repositorio: `financieras/antigen_predictor`
4. En **"Branch"** selecciona `main`
5. En **"Main file path"** escribe: `app/app.py`
6. Haz clic en **"Deploy"**

Streamlit tardará 1-3 minutos en instalar dependencias y arrancar la app.  
La URL resultante (`https://xxxx.streamlit.app`) es permanente y compartible.

### 3.4 Actualizar la app desplegada

Cualquier `git push` al repositorio actualiza automáticamente la app desplegada.
No hay que hacer nada extra en Streamlit Cloud.

---
## 4. Ajuste de rutas en app.py para Streamlit Community Cloud

En local, `app.py` carga el modelo con `joblib.load("model.pkl")`,
lo que funciona porque el directorio de trabajo es `app/`.

En Streamlit Community Cloud el directorio de trabajo es la **raíz del repositorio**,
así que la ruta correcta es `models/model.pkl`.

La celda siguiente actualiza `app.py` para que funcione en ambos entornos:

In [4]:
app_path = PROJECT_ROOT / 'app' / 'app.py'
content  = app_path.read_text()

# Reemplazar la carga del modelo por una versión compatible con ambos entornos
old_load = 'bundle = joblib.load("model.pkl")'
new_load = '''# Ruta compatible con ejecución local (app/) y Streamlit Community Cloud (raíz)
from pathlib import Path
_here = Path(__file__).parent          # carpeta donde está app.py
_root = _here.parent                   # raíz del repositorio
_model_path = _root / "models" / "model.pkl"
if not _model_path.exists():
    _model_path = _here / "model.pkl"  # fallback para compatibilidad
bundle = joblib.load(_model_path)'''

if old_load in content:
    content = content.replace(old_load, new_load)
    app_path.write_text(content)
    print("✓ app.py actualizado con ruta de modelo compatible.")
elif '_model_path' in content:
    print("✓ app.py ya tiene la ruta de modelo compatible.")
else:
    print("⚠️  No se encontró el patrón esperado en app.py.")
    print("   Revisa manualmente la línea de carga del modelo.")

✓ app.py actualizado con ruta de modelo compatible.


---
## 5. Commit y push final

Una vez verificado todo, sube los cambios a GitHub para que Streamlit Cloud los recoja:

In [ ]:
import subprocess

def git(cmd):
    result = subprocess.run(
        cmd, cwd=str(PROJECT_ROOT),
        capture_output=True, text=True
    )
    output = result.stdout.strip() or result.stderr.strip()
    if output:
        print(output)
    return result.returncode

print("Estado del repositorio:")
git(['git', 'status', '--short'])

print("\nArchivos pendientes de commit:")
git(['git', 'diff', '--name-only', 'HEAD'])

In [ ]:
# Ejecuta esta celda para hacer commit y push de todos los cambios
# Edita el mensaje de commit si lo necesitas

COMMIT_MSG = "Update app.py model path for Streamlit Community Cloud"

git(['git', 'add', '-A'])
git(['git', 'commit', '-m', COMMIT_MSG])
git(['git', 'push'])
print("\nCambios subidos a GitHub. Streamlit Cloud actualizará la app automáticamente.")

---
## 6. Checklist de la presentación

Antes del día de la presentación verifica que:

- [ ] La app está desplegada y accesible en `https://xxxx.streamlit.app`
- [ ] El botón **"Cargar proteínas SARS-CoV-2"** funciona
- [ ] Spike glycoprotein aparece con el score más alto
- [ ] La descarga CSV funciona
- [ ] La URL es accesible desde otro dispositivo (móvil o portátil distinto)
- [ ] El sidebar muestra AUC-ROC: 0.719 y Proteínas de entrenamiento: 1,310

Durante la presentación puedes invitar a los asistentes a abrir la URL
en sus propios portátiles y probar el modelo con sus propias secuencias.